# PDI: Previsão de Inadimplência de Cartão de Crédito
**Autor:** Francisco Bruno Lopes Grangeiro

## 1. Descrição do Problema e dos Dados
Neste projeto, atuo como Engenheiro de Machine Learning e o meu objetivo é construir, avaliar e interpretar modelos supervisionados. O problema prático que busco resolver é a **previsão de inadimplência (default)** de clientes de cartão de crédito em Taiwan.

**Motivação para o uso de Machine Learning:** A decisão de crédito baseada em regras fixas, heurísticas simples ou análises estatísticas tradicionais não é suficiente. O comportamento financeiro humano é altamente não-linear e multivariado. Um limite alto pode significar segurança, mas se acompanhado de um padrão recente de atrasos, representa risco iminente. O Machine Learning permite extrair essas relações complexas em larga escala.

**Desafios do Domínio:** O principal desafio é o desbalanceamento inerente de classes — a vasta maioria dos clientes honra as dívidas. Há também um custo fortemente assimétrico: libertar crédito a um mau pagador causa prejuízo direto massivo. Isto exige alta revocação (*Recall*). Além disso, existe a necessidade de explicabilidade, pois as recusas de crédito frequentemente exigem amparo regulatório.

**Descrição Técnica do Dataset:**
- **Origem dos dados:** Clientes de cartão de crédito de uma instituição financeira em Taiwan.
- **Volume:** Aproximadamente 30.000 observações.
- **Natureza das variáveis:** Variáveis demográficas categóricas já codificadas (Género, Educação, Estado Civil), atributos contínuos (Idade, Limite de Crédito, Valores de Fatura e Pagamento) e atributos ordinais/categóricos (Histórico de pagamento de abril a setembro de 2005).
- **Variável-alvo:** `default payment next month` (Classificação binária: 1 para risco/inadimplente, 0 para pagador). É formulada como binária para gerar limites práticos e acionar fluxos de aprovação/rejeição no sistema.
- **Features utilizadas:** O histórico de pagamento atua como *proxy* temporal crucial da situação atual do cliente. As variáveis demográficas fornecem contextos de risco complementares.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Dicionário para guardar os resultados de todos os modelos
resultados_modelos = {}

# Função auxiliar para calcular métricas e desenhar a Matriz de Confusão
def avaliar_modelo(nome_modelo, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    resultados_modelos[nome_modelo] = {'Acurácia': acc, 'Precisão': prec, 'Recall': rec, 'F1-Score': f1}
    
    print(f"--- Modelo: {nome_modelo} ---")
    print(f"Acurácia:  {acc:.4f}")
    print(f"Precisão:  {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}\n")
    
    # Matriz de Confusão
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
                xticklabels=['Pagador (0)', 'Inadimplente (1)'], 
                yticklabels=['Pagador (0)', 'Inadimplente (1)'])
    plt.title(f'Matriz de Confusão - {nome_modelo}')
    plt.xlabel('Previsão do Modelo')
    plt.ylabel('Valor Real')
    plt.show()

## 2. Carregamento e Preparação dos Dados
Nesta etapa, carrego a base de dados bruta, removo identificadores sem poder preditivo e separo as variáveis preditoras (X) da variável-alvo (y).

Utilizo a divisão estratificada (`stratify=y`) devido ao forte desbalanceamento, garantindo que os conjuntos de Treino e Teste mantenham as proporções originais da população. Também aplico a padronização matemática (`StandardScaler`), um requisito essencial para modelos baseados em geometria e distâncias lineares.

In [ ]:
# Carrega os dados da folha Excel (ignorando a primeira linha de cabeçalho duplo)
df = pd.read_excel('assets/default_credit_card_clients.xls', header=1)

# Separando X (features) e y (target) - Removendo ID
df = df.drop(columns=['ID']) 
X = df.drop(columns=['default payment next month'])
y = df['default payment next month']

# Divisão em Treino e Teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Escalonamento (Necessário para o Perceptron)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamanho do Treino: {X_train.shape[0]} amostras")
print(f"Tamanho do Teste: {X_test.shape[0]} amostras")

## 3. Modelo Baseline: Classificador Linear com Perceptron
Para entender o comportamento geométrico do problema e estabelecer um ponto de comparação claro, começo com o **Perceptron**. 
Ele tenta separar os inadimplentes dos pagadores criando um hiperplano reto no espaço multivariado. Isso indicar-me-á de forma explícita as limitações de usar separações puramente lineares num problema de comportamento humano.

In [ ]:
# Treinando o Perceptron
clf_perc = Perceptron(random_state=42, max_iter=1000, tol=1e-3)
clf_perc.fit(X_train_scaled, y_train)
y_pred_perc = clf_perc.predict(X_test_scaled)

avaliar_modelo("Perceptron Linear (Baseline)", y_test, y_pred_perc)

In [ ]:
# Extraindo os pesos (coeficientes) e o bias do Perceptron para interpretação
pesos = clf_perc.coef_[0]
bias = clf_perc.intercept_[0]

print(f"Bias (Intercepto/w0): {bias:.4f}\n")

print("Top 3 variáveis que mais inclinam para INADIMPLÊNCIA (pesos positivos):")
indices_pos = np.argsort(pesos)[-3:][::-1]
for i in indices_pos:
    print(f"- {X.columns[i]}: {pesos[i]:.4f}")

print("\nTop 3 variáveis que mais inclinam para PAGAMENTO (pesos negativos):")
indices_neg = np.argsort(pesos)[:3]
for i in indices_neg:
    print(f"- {X.columns[i]}: {pesos[i]:.4f}")

**Interpretação do Perceptron (Hiperplano e Bias):**
- **Coeficientes e Hiperplano:** A equação geométrica (`ŷ = w₀ + w₁x₁ ...`) dita a orientação da fronteira de decisão. O modelo aprendeu que o atraso nos meses mais recentes (`PAY_0`, `PAY_2`) possui os maiores coeficientes positivos, empurrando a decisão para o lado "Inadimplente". Em contrapartida, altos limites de crédito (`LIMIT_BAL`) e históricos mais antigos (`PAY_5`) possuem pesos negativos, puxando a decisão para a classe negativa ("Pagador").
- **O Papel do Bias (w₀):** O *bias* (viés) corrige o ponto de origem do hiperplano no espaço vetorial. Como a maioria esmagadora dos clientes na nossa base é de bons pagadores, o viés ajusta a fronteira de forma a compensar esse desbalanceamento natural da equação.
- **Limitações e Indícios de Underfitting:** O baixo F1-score e um *Recall* sofrível evidenciam o *underfitting*. O problema não é linearmente separável (uma linha reta é ineficaz para captar nuances como "alto limite, mas desempregado recentemente"). O Perceptron baseia a sua atualização estritamente em erros de classificação, o que o torna altamente instável e sensível ao ruído e a *outliers* do mundo financeiro.

## 4. Modelo com Árvore de Decisão
Como o comportamento de crédito é repleto de nuances condicionais e não-lineares, a evolução lógica é testar uma **Árvore de Decisão**.
Abaixo, treino o modelo com parâmetros padrão (arriscando o *overfitting* devido à extrema flexibilidade da árvore) e, logo a seguir, configuro a Validação Cruzada (GridSearch) para impor podas de regularização e garantir a generalização.

In [ ]:
# 1. Árvore Padrão (Sem poda, alta flexibilidade e risco de Overfitting)
clf_dt_default = DecisionTreeClassifier(random_state=42)
clf_dt_default.fit(X_train, y_train) # Sem dados escalonados (árvores não requerem isso)
y_pred_dt_default = clf_dt_default.predict(X_test)
avaliar_modelo("Árvore de Decisão (Padrão)", y_test, y_pred_dt_default)

# 2. Validação Cruzada e Busca Sistemática de Hiperparâmetros
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid_dt = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 5, 20]
}

# Otimizando para a métrica F1 devido ao desbalanceamento
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=cv, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)
y_pred_dt_cv = grid_dt.predict(X_test)

avaliar_modelo("Árvore de Decisão (Otimizada)", y_test, y_pred_dt_cv)
print(f"Melhores hiperparâmetros (DT): {grid_dt.best_params_}\n")

# Representação visual das regras extraídas pelo modelo otimizado
plt.figure(figsize=(16, 8))
plot_tree(grid_dt.best_estimator_, feature_names=X.columns, class_names=['Pagador', 'Inadimplente'], 
          filled=True, rounded=True, max_depth=2, fontsize=10)
plt.title("Fronteira Explícita: Regras da Árvore de Decisão Otimizada (Top 2 Níveis)")
plt.show()

In [ ]:
# Analisando a robustez e variação entre os Folds da Validação Cruzada
resultados_cv = pd.DataFrame(grid_dt.cv_results_)
melhor_indice = grid_dt.best_index_

f1_medio = resultados_cv.loc[melhor_indice, 'mean_test_score']
f1_desvio = resultados_cv.loc[melhor_indice, 'std_test_score']

print(f"F1-Score Médio na Validação Cruzada (Treino): {f1_medio:.4f}")
print(f"Variação (Desvio Padrão) entre os 5 Folds: +/- {f1_desvio:.4f}")

**Discussão sobre Validação e Regularização (Árvores):**
- **Interpretação das Regras:** O caminho do nó raiz até às folhas traduz perfeitamente o conhecimento do domínio. A divisão principal (`PAY_0 <= 1.5`) revela logo de início que clientes com atrasos severos no último mês observado devem ser isolados num ramo de alto risco.
- **Risco de Overfitting e Mudança na Interpretabilidade:** A árvore padrão decorou o ruído, gerando quebras excessivamente específicas (o seu *F1-score* no teste afundou para 0.39). Após aplicar a regularização por meio do `GridSearch` (limitando a profundidade a 3), a árvore reduziu a complexidade, melhorou a generalização (*F1-score* de 0.46) e tornou-se subitamente muito mais fácil de interpretar e auditar por equipas de negócio.
- **Robustez e Escolha do GridSearch:** Como o desvio padrão do *F1-Score* variou de forma minúscula (+/- 0.0151) entre os 5 *folds*, a estabilidade das partições está comprovada. Optei pelo *GridSearch* em detrimento do *Random Search* pois o meu espaço de busca era delimitado computacionalmente, garantindo que o algoritmo varresse todas as combinações sem o risco de ficar retido em "mínimos locais".

## 5. Modelos Avançados: Ensemble via Random Forest
Para tentar suplantar a limitação de uma árvore individual sem sacrificar a sua natureza não-linear, aplico agora uma abordagem de *Ensemble* baseada em *Bagging*: a **Random Forest**.

O objetivo é avaliar o ganho de capacidade representacional gerado pela votação de múltiplas árvores e verificar como podemos manter a explicabilidade através do mapeamento da importância de cada variável de negócio.

In [ ]:
# Otimização do Random Forest via Validação Cruzada
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10],
    'min_samples_leaf': [5, 10]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=cv, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)
y_pred_rf = grid_rf.predict(X_test)

avaliar_modelo("Random Forest (Otimizado)", y_test, y_pred_rf)
print(f"Melhores hiperparâmetros (RF): {grid_rf.best_params_}")

# Extraindo e interpretando a Importância das Features do Ensemble
best_rf = grid_rf.best_estimator_
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]

top_n = 10
top_indices = indices[:top_n]
top_features = [X.columns[i] for i in top_indices]
top_importances = importances[top_indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=top_importances, y=top_features, palette="viridis")
plt.title("Interpretação: Top 10 Variáveis Mais Importantes (Random Forest)")
plt.xlabel("Importância Relativa (Ganho de Informação)")
plt.ylabel("Variáveis")
plt.show()

**Análise da Random Forest (Fronteira, Complexidade e Custo):**
- **Fronteira de Decisão Agregada:** Diferentemente do Perceptron (linha rígida) e da Árvore Simples (cortes ortogonais abruptos), a Random Forest traça uma fronteira muito mais difusa e robusta. Sendo um *ensemble*, compensa os erros isolados de uma árvore através do voto maioritário, estabilizando as predições.
- **Importância das Features:** Mesmo sem o desenho óbvio de uma única árvore, garantimos a interpretabilidade provando que quase 30% da força de decisão do modelo provém apenas do `PAY_0` (estado de pagamento do mês mais recente). As demografias importam muito menos do que o comportamento histórico.
- **Desempenho vs Custo Computacional:** Houve um aumento notável do custo computacional de treino devido ao *GridSearch* cruzado com 100 estimadores. Todavia, num contexto bancário *batch* (processamento noturno), a latência do treino não é impeditiva. O ganho de generalização justifica plenamente o investimento computacional.

## 6. Comparação Global de Desempenho
Abaixo ploto um consolidado métrico provando o avanço na otimização em relação ao modelo original linear.

In [ ]:
# Agrupando os resultados para visualização gráfica
df_resultados = pd.DataFrame(resultados_modelos).T

df_resultados.plot(kind='bar', figsize=(12, 6), colormap='Set2', edgecolor='black')
plt.title("Comparação de Desempenho dos Modelos de Machine Learning", fontsize=14)
plt.ylabel("Pontuação (0 a 1)")
plt.xlabel("Modelos")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.ylim(0, 1.0) 

for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Comparação Probabilística (Curva ROC)
Para avaliar os modelos em múltiplos limiares e não apenas na separação seca de 50%, analisamos a Curva ROC e a métrica global *AUC* (Área sob a Curva).

In [ ]:
# Desenhando a Curva ROC comparativa
plt.figure(figsize=(10, 8))

# 1. Perceptron (Utiliza decision_function, já que não emite predict_proba)
y_score_perc = clf_perc.decision_function(X_test_scaled)
fpr_perc, tpr_perc, _ = roc_curve(y_test, y_score_perc)
plt.plot(fpr_perc, tpr_perc, label=f'Perceptron Linear (AUC = {auc(fpr_perc, tpr_perc):.2f})')

# 2. Árvore de Decisão Padrão
y_score_dt_def = clf_dt_default.predict_proba(X_test)[:, 1]
fpr_dt_def, tpr_dt_def, _ = roc_curve(y_test, y_score_dt_def)
plt.plot(fpr_dt_def, tpr_dt_def, label=f'Árvore Padrão (AUC = {auc(fpr_dt_def, tpr_dt_def):.2f})')

# 3. Árvore Otimizada
y_score_dt_cv = grid_dt.predict_proba(X_test)[:, 1]
fpr_dt_cv, tpr_dt_cv, _ = roc_curve(y_test, y_score_dt_cv)
plt.plot(fpr_dt_cv, tpr_dt_cv, label=f'Árvore Otimizada (AUC = {auc(fpr_dt_cv, tpr_dt_cv):.2f})', linewidth=2)

# 4. Random Forest
y_score_rf = grid_rf.predict_proba(X_test)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_score_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc(fpr_rf, tpr_rf):.2f})', linewidth=2)

# Chute aleatório (Baseline 0.5)
plt.plot([0, 1], [0, 1], 'k--', label='Chute Aleatório (AUC = 0.50)')

plt.title('Curva ROC - Comparação Geral dos Modelos', fontsize=14)
plt.xlabel('Taxa de Falsos Positivos (FPR) - "Pagadores incorretamente penalizados"', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (TPR) - "Inadimplentes corretos"', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Discussão Aplicada e Conclusões Finais
Para finalizar, sintetizo a viabilidade deste projeto no quotidiano organizacional sob a perspetiva de um Engenheiro de Machine Learning:

1. **Viabilidade do modelo no contexto real:**
Ambos os modelos baseados em árvores (DT otimizada e *Random Forest*) mostraram-se tecnicamente viáveis e seriam passíveis de *deploy* imediato. Modelos assentes nestas topologias são de baixíssima latência na inferência (prevêem em frações de segundo), o que atende plenamente à necessidade transacional e aos sistemas de tomada de decisão (*Credit Scoring Engine*) instantânea de um banco moderno.

2. **Limitações técnicas e operacionais:**
O gargalo principal repousa no viés gerado pelo *desbalanceamento nativo* do *dataset*. Mesmo após as otimizações sistémicas, o nosso *Recall* estabilizou-se perto dos 35-36% na deteção da classe minoritária. Em termos operacionais, isto representa uma alta taxa de "Falsos Negativos" (o modelo acaba por classificar demasiados devedores como sendo aptos).

3. **Possíveis melhorias e extensões futuras:**
- **Lidar com o desbalanceamento ao nível dos dados e algoritmos:** A primeira extensão mandatória será introduzir matrizes de custo (`class_weight='balanced'`) no interior dos algoritmos ou técnicas de sobreamostragem da minoria, como o algoritmo SMOTE.
- **Ajuste flexível do Threshold de Risco:** Baseado na Curva ROC, que demonstrou a superioridade probabilística da Random Forest, podemos baixar o limiar de corte de 50% para, digamos, 25% de probabilidade. Isso forçaria o modelo a rotular o cliente como "Inadimplente" mais facilmente, abdicando de Precisão mas mitigando perdas massivas de capital através de um *Recall* robusto.
- **Engenharia de Features:** Poderíamos criar métricas combinadas (ex: "Montante da Fatura vs Limite Global") em vez de alimentar apenas os valores absolutos.